# SOC+NMR model coefficients

Coefficients at `lambda.min` from the SOC+NMR models saved with `fit_full = TRUE` in `outputs/main/`. Clinical covariates are listed first, then Nightingale biomarkers in `rowdata` order.


In [ ]:
library(glmnet)
library(tidyverse)
library(here)

In [ ]:
feature_order <- c(
  "age_at_sample",
  "sex",
  "BMI_interp",
  "SBP_interp",
  "M_a1c",
  "A1C_obs",
  "T_a1c",
  "M_egfr",
  "EGFR_obs",
  "T_egfr",
  "M_fib4",
  "FIB4_obs",
  "T_fib4",
  "M_tot_choles",
  "TOT_CHOLES_obs",
  "T_tot_choles",
  "M_hdl_choles",
  "HDL_CHOLES_obs",
  "T_hdl_choles",
  "M_smoke",
  "smk_current",
  "smk_former",
  "diabetes_at_baseline",
  "bp_treatment",
  "statin_treatment",
  "smoking_current",
  "M_smoke_af",
  "smoking_current_af",
  "race",
  "imputed_Diastolic",
  "DBP_interp",
  "prevalent_hf_cm",
  "hf_at_baseline",
  "prevalent_mi",
  "mi_at_baseline",
  "imputed_height_m",
  "height_interp",
  "imputed_weight_kg",
  "weight_interp"
)

feature_labels <- c(
  age_at_sample        = "Age at enrollment",
  sex                  = "Sex (Male = 1, Female = 0)",
  BMI_interp           = "Body-Mass-Index",
  SBP_interp           = "Systolic Blood Pressure",
  M_a1c                = "Missingness HbA1c indicator",
  A1C_obs              = "Observed HbA1c value",
  T_a1c                = "Time delay HbA1c measurement",
  M_egfr               = "Missingness eGFR indicator",
  EGFR_obs             = "Observed eGFR value",
  T_egfr               = "Time delay eGFR measurement",
  M_fib4               = "Missingness FIB-4 indicator",
  FIB4_obs             = "Observed FIB-4 value",
  T_fib4               = "Time delay FIB-4 measurement",
  M_tot_choles         = "Missingness Total-Chol indicator",
  TOT_CHOLES_obs       = "Observed Total-Chol value",
  T_tot_choles         = "Time delay Total-Chol measurement",
  M_hdl_choles         = "Missingness HDL-C indicator",
  HDL_CHOLES_obs       = "Observed HDL-C value",
  T_hdl_choles         = "Time delay HDL-C measurement",
  M_smoke              = "Missingness Smoking indicator",
  smk_current          = "Smoking reported at enrollment",
  smk_former           = "Former Smoking reported at enrollment",
  diabetes_at_baseline = "Prior diabetes dx at enrollment",
  bp_treatment         = "Blood pressure rx at time of enrollment",
  statin_treatment     = "Statin rx at time of enrollment",
  smoking_current      = "Current smoking (CHARGE-AF)",
  M_smoke_af           = "Missingness smoking indicator (CHARGE-AF)",
  smoking_current_af   = "Current smoking (CHARGE-AF)",
  race                 = "Race (White = 1, Non-White = 0)",
  imputed_Diastolic    = "Diastolic Blood Pressure",
  DBP_interp           = "Diastolic Blood Pressure",
  prevalent_hf_cm      = "Prior heart failure dx at enrollment",
  hf_at_baseline       = "Prior heart failure dx at enrollment",
  prevalent_mi         = "Prior myocardial infarction dx at enrollment",
  mi_at_baseline       = "Prior myocardial infarction dx at enrollment",
  imputed_height_m     = "Height at enrollment",
  height_interp        = "Height at enrollment",
  imputed_weight_kg    = "Weight at enrollment",
  weight_interp        = "Weight at enrollment"
)


In [ ]:
model_dir <- here("outputs/main")

# SOC+NMR fits saved with fit_full = TRUE
model_files <- c(
  T2D             = "t2d_a1c_marker_nmr_oof.rds",
  CKD             = "ckd_egfr_marker_nmr_oof.rds",
  MASLD           = "nafld_fib4_marker_nmr_oof.rds",
  `Heart Failure` = "hf_cm_prevent_ingredients_marker_nmr_oof.rds",
  `Acute MI`      = "acute_mi_prevent_ingredients_marker_nmr_oof.rds",
  CHD             = "chd_revasc_prevent_ingredients_marker_nmr_oof.rds",
  `TIA/Stroke`    = "tia_stroke_prevent_ingredients_marker_nmr_oof.rds",
  `Total CVD`     = "total_cvd_prevent_ingredients_marker_nmr_oof.rds",
  AFib            = "afib_chargeaf_ingredients_marker_nmr_oof.rds"
)

extract_glmnet_coef <- function(model) {
  coef(model$full_model$model$glmnet.fit, s = model$full_model$model$lambda.min) %>%
    as.matrix() %>%
    as.data.frame() %>%
    tibble::rownames_to_column("feature") %>%
    dplyr::rename(coef = 2) %>%
    filter(feature != "(Intercept)")
}

coef_list <- lapply(names(model_files), function(label) {
  extract_glmnet_coef(readRDS(file.path(model_dir, model_files[[label]]))) %>%
    select(feature, !!label := coef)
})

joined <- purrr::reduce(coef_list, full_join, by = "feature")

rowdata <- read.csv(here("data/rowdata.csv")) %>%
  distinct(name, Excel_column_name, Biomarker_name)

nmr_features <- intersect(rowdata$name, joined$feature)
soc_features <- setdiff(joined$feature, nmr_features)
row_order <- c(
  intersect(feature_order, soc_features),
  setdiff(soc_features, feature_order),
  nmr_features
)

joined <- joined %>%
  mutate(feature = factor(feature, levels = row_order)) %>%
  arrange(feature) %>%
  mutate(feature = as.character(feature)) %>%
  left_join(rowdata, by = c("feature" = "name")) %>%
  mutate(
    `Feature Name` = coalesce(
      unname(feature_labels[feature]),
      Excel_column_name,
      feature
    )
  ) %>%
  select(
    Feature = feature,
    `Feature Name`,
    `Biomarker Name` = Biomarker_name,
    everything(),
    -Excel_column_name
  )

joined %>% head()

out_dir <- here("outputs/tables")
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
write.csv(joined, file.path(out_dir, "PRIMARY_COEFF.csv"), row.names = FALSE)